In [1]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer
from IPython.display import display

In [2]:
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "insurance_documents"

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = client.get_collection(
    name=COLLECTION_NAME
)

print("Collection connected successfully!")
print("Collection name:", collection.name)
print("Total chunks:", collection.count())

Collection connected successfully!
Collection name: insurance_documents
Total chunks: 872


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [4]:
sample = collection.get(limit=3)

for i in range(len(sample["ids"])):
    print("\nChunk ID:", sample["ids"][i])
    print("Metadata:", sample["metadatas"][i])
    print("Text:", sample["documents"][i][:500])
    print("-" * 80)


Chunk ID: chunk_0
Metadata: {'page': 1, 'document_type': 'Policy Usage Guide', 'source': 'care--policy-usage-guide.pdf'}
Text: Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Ca
--------------------------------------------------------------------------------

Chunk ID: chunk_1
Metadata: {'document_type': 'Policy Usage Guide', 'page': 1, 'source': 'care--policy-usage-guide.pdf'}
Text: We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical 

In [5]:
def retrieve_chunks(question, n_results=5):
    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    rows = []

    for i in range(len(results["ids"][0])):
        rows.append({
            "rank": i + 1,
            "chunk_id": results["ids"][0][i],
            "metadata": results["metadatas"][0][i],
            "text": results["documents"][0][i],
            "distance": results["distances"][0][i]
        })

    return pd.DataFrame(rows)

In [7]:
questions = [
    "What qualifies as an inpatient hospitalization?",
    "What is a day care treatment?",
    "How long are pre-hospitalization expenses covered?",
    "How long are post-hospitalization expenses covered?",
    "Is emergency ambulance expense covered?"
]

all_results = {}

for i, question in enumerate(questions, start=1):
    print(f"\nQUESTION {i}: {question}")

    df = retrieve_chunks(question, n_results=5)
    all_results[f"Q{i}"] = df

    display(df[["rank", "chunk_id", "metadata", "text", "distance"]])


QUESTION 1: What qualifies as an inpatient hospitalization?


,rank,chunk_id,metadata,text,distance
0,1,chunk_503,{'document_type': 'Prospectus / Sales Literatu...,which forms a part of Hospitalization. (ii) Da...,0.894764
1,2,chunk_577,"{'page': 10, 'document_type': 'Prospectus / Sa...",It all adds up - A trip to a hospital involves...,0.915413
2,3,chunk_0,"{'document_type': 'Policy Usage Guide', 'page'...",Policy Usage Guide In-patient Hospitalization ...,0.927046
3,4,chunk_29,{'source': 'care-freedom-(health-insurance-pro...,"and treatment at a hospital), hospitalisation ...",0.931150
4,5,chunk_507,"{'page': 2, 'document_type': 'Prospectus / Sal...","Hospitalization, provided that We shall not be...",0.937506



QUESTION 2: What is a day care treatment?


,rank,chunk_id,metadata,text,distance
0,1,chunk_503,"{'page': 2, 'source': 'care-plus--(health-insu...",which forms a part of Hospitalization. (ii) Da...,0.915332
1,2,chunk_163,{'source': 'care-freedom-(health-insurance-pro...,for personal care such as help with activities...,0.938073
2,3,chunk_0,"{'page': 1, 'source': 'care--policy-usage-guid...",Policy Usage Guide In-patient Hospitalization ...,0.955394
3,4,chunk_850,"{'page': 1, 'document_type': 'Policy Usage Gui...","such as room charges, nursing expenses, intens...",0.992809
4,5,chunk_502,"{'page': 2, 'source': 'care-plus--(health-insu...","care due to Illness or Injury , which should b...",0.993983



QUESTION 3: How long are pre-hospitalization expenses covered?


,rank,chunk_id,metadata,text,distance
0,1,chunk_506,"{'page': 2, 'document_type': 'Prospectus / Sal...",Expenses: (i) Pre-Hospitalization Medical Expe...,0.438086
1,2,chunk_851,"{'document_type': 'Policy Usage Guide', 'sourc...",for expenses incurred under Benefit 'In-patien...,0.492926
2,3,chunk_200,{'document_type': 'Prospectus / Sales Literatu...,Hospitalization expenses valid till 30 days be...,0.570823
3,4,chunk_72,{'document_type': 'Prospectus / Sales Literatu...,to a Hospital for treatment of any One Illness...,0.573324
4,5,chunk_1,"{'document_type': 'Policy Usage Guide', 'page'...",We cover more than 540 such treatments. Some o...,0.586690



QUESTION 4: How long are post-hospitalization expenses covered?


,rank,chunk_id,metadata,text,distance
0,1,chunk_200,"{'page': 18, 'document_type': 'Prospectus / Sa...",Hospitalization expenses valid till 30 days be...,0.512936
1,2,chunk_1,"{'page': 1, 'document_type': 'Policy Usage Gui...",We cover more than 540 such treatments. Some o...,0.595161
2,3,chunk_212,{'source': 'care-freedom-(health-insurance-pro...,"10 days Rs.15,000 if Hospitalization exceeds 1...",0.596087
3,4,chunk_75,{'source': 'care-freedom-(health-insurance-pro...,- perhaps Your state of health is such that Yo...,0.617227
4,5,chunk_72,{'source': 'care-freedom-(health-insurance-pro...,to a Hospital for treatment of any One Illness...,0.620908



QUESTION 5: Is emergency ambulance expense covered?


,rank,chunk_id,metadata,text,distance
0,1,chunk_11,"{'document_type': 'Policy Usage Guide', 'page'...","fees, prescribed diagnostics or prescribed med...",0.679911
1,2,chunk_28,{'source': 'care-freedom-(health-insurance-pro...,COVER Getting to the hospital - It is our utmo...,0.680229
2,3,chunk_510,{'source': 'care-plus--(health-insurance-produ...,"especially in an emergency. Towards that end, ...",0.736925
3,4,chunk_74,{'document_type': 'Prospectus / Sales Literatu...,3 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 5....,0.778727
4,5,chunk_27,"{'page': 3, 'source': 'care-freedom-(health-in...",POST-HOSPITALISATION Back home and till you ar...,0.794325


In [6]:
question = "What is a day care treatment?"

results_df = retrieve_chunks(question, n_results=5)

display(results_df)

,rank,chunk_id,metadata,text,distance
0,1,chunk_503,{'source': 'care-plus--(health-insurance-produ...,which forms a part of Hospitalization. (ii) Da...,0.915332
1,2,chunk_163,{'document_type': 'Prospectus / Sales Literatu...,for personal care such as help with activities...,0.938073
2,3,chunk_0,"{'page': 1, 'source': 'care--policy-usage-guid...",Policy Usage Guide In-patient Hospitalization ...,0.955394
3,4,chunk_850,"{'page': 1, 'source': 'care-supreme----policy-...","such as room charges, nursing expenses, intens...",0.992809
4,5,chunk_502,"{'page': 2, 'source': 'care-plus--(health-insu...","care due to Illness or Injury , which should b...",0.993983


In [9]:
for qid, df in all_results.items():
    print(f"\n{qid}")
    display(df[["rank", "chunk_id", "metadata", "text"]])


Q1


,rank,chunk_id,metadata,text
0,1,chunk_503,{'document_type': 'Prospectus / Sales Literatu...,which forms a part of Hospitalization. (ii) Da...
1,2,chunk_577,"{'page': 10, 'document_type': 'Prospectus / Sa...",It all adds up - A trip to a hospital involves...
2,3,chunk_0,"{'document_type': 'Policy Usage Guide', 'page'...",Policy Usage Guide In-patient Hospitalization ...
3,4,chunk_29,{'source': 'care-freedom-(health-insurance-pro...,"and treatment at a hospital), hospitalisation ..."
4,5,chunk_507,"{'page': 2, 'document_type': 'Prospectus / Sal...","Hospitalization, provided that We shall not be..."



Q2


,rank,chunk_id,metadata,text
0,1,chunk_503,"{'page': 2, 'source': 'care-plus--(health-insu...",which forms a part of Hospitalization. (ii) Da...
1,2,chunk_163,{'source': 'care-freedom-(health-insurance-pro...,for personal care such as help with activities...
2,3,chunk_0,"{'page': 1, 'source': 'care--policy-usage-guid...",Policy Usage Guide In-patient Hospitalization ...
3,4,chunk_850,"{'page': 1, 'document_type': 'Policy Usage Gui...","such as room charges, nursing expenses, intens..."
4,5,chunk_502,"{'page': 2, 'source': 'care-plus--(health-insu...","care due to Illness or Injury , which should b..."



Q3


,rank,chunk_id,metadata,text
0,1,chunk_506,"{'page': 2, 'document_type': 'Prospectus / Sal...",Expenses: (i) Pre-Hospitalization Medical Expe...
1,2,chunk_851,"{'document_type': 'Policy Usage Guide', 'sourc...",for expenses incurred under Benefit 'In-patien...
2,3,chunk_200,{'document_type': 'Prospectus / Sales Literatu...,Hospitalization expenses valid till 30 days be...
3,4,chunk_72,{'document_type': 'Prospectus / Sales Literatu...,to a Hospital for treatment of any One Illness...
4,5,chunk_1,"{'document_type': 'Policy Usage Guide', 'page'...",We cover more than 540 such treatments. Some o...



Q4


,rank,chunk_id,metadata,text
0,1,chunk_200,"{'page': 18, 'document_type': 'Prospectus / Sa...",Hospitalization expenses valid till 30 days be...
1,2,chunk_1,"{'page': 1, 'document_type': 'Policy Usage Gui...",We cover more than 540 such treatments. Some o...
2,3,chunk_212,{'source': 'care-freedom-(health-insurance-pro...,"10 days Rs.15,000 if Hospitalization exceeds 1..."
3,4,chunk_75,{'source': 'care-freedom-(health-insurance-pro...,- perhaps Your state of health is such that Yo...
4,5,chunk_72,{'source': 'care-freedom-(health-insurance-pro...,to a Hospital for treatment of any One Illness...



Q5


,rank,chunk_id,metadata,text
0,1,chunk_11,"{'document_type': 'Policy Usage Guide', 'page'...","fees, prescribed diagnostics or prescribed med..."
1,2,chunk_28,{'source': 'care-freedom-(health-insurance-pro...,COVER Getting to the hospital - It is our utmo...
2,3,chunk_510,{'source': 'care-plus--(health-insurance-produ...,"especially in an emergency. Towards that end, ..."
3,4,chunk_74,{'document_type': 'Prospectus / Sales Literatu...,3 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 5....
4,5,chunk_27,"{'page': 3, 'source': 'care-freedom-(health-in...",POST-HOSPITALISATION Back home and till you ar...


In [8]:
retrieval_rows = []

for question_id, df in all_results.items():
    for _, row in df.iterrows():
        retrieval_rows.append({
            "question_id": question_id,
            "rank": row["rank"],
            "chunk_id": row["chunk_id"],
            "metadata": str(row["metadata"]),
            "text": row["text"],
            "distance": row["distance"]
        })

retrieval_df = pd.DataFrame(retrieval_rows)

retrieval_df.to_csv(
    "retrieval_results.csv",
    index=False
)

print("Saved as retrieval_results.csv")
display(retrieval_df)

Saved as retrieval_results.csv


,question_id,rank,chunk_id,metadata,text,distance
0,Q1,1,chunk_503,{'document_type': 'Prospectus / Sales Literatu...,which forms a part of Hospitalization. (ii) Da...,0.894764
1,Q1,2,chunk_577,"{'page': 10, 'document_type': 'Prospectus / Sa...",It all adds up - A trip to a hospital involves...,0.915413
2,Q1,3,chunk_0,"{'document_type': 'Policy Usage Guide', 'page'...",Policy Usage Guide In-patient Hospitalization ...,0.927046
3,Q1,4,chunk_29,{'source': 'care-freedom-(health-insurance-pro...,"and treatment at a hospital), hospitalisation ...",0.931150
4,Q1,5,chunk_507,"{'page': 2, 'document_type': 'Prospectus / Sal...","Hospitalization, provided that We shall not be...",0.937506
5,Q2,1,chunk_503,"{'page': 2, 'source': 'care-plus--(health-insu...",which forms a part of Hospitalization. (ii) Da...,0.915332
6,Q2,2,chunk_163,{'source': 'care-freedom-(health-insurance-pro...,for personal care such as help with activities...,0.938073
7,Q2,3,chunk_0,"{'page': 1, 'source': 'care--policy-usage-guid...",Policy Usage Guide In-patient Hospitalization ...,0.955394
8,Q2,4,chunk_850,"{'page': 1, 'document_type': 'Policy Usage Gui...","such as room charges, nursing expenses, intens...",0.992809
9,Q2,5,chunk_502,"{'page': 2, 'source': 'care-plus--(health-insu...","care due to Illness or Injury , which should b...",0.993983


In [10]:
q1_df = all_results["Q1"]

for _, row in q1_df.iterrows():
    print("\nRank:", row["rank"])
    print("Chunk ID:", row["chunk_id"])
    print("Metadata:", row["metadata"])
    print("Full text:\n", row["text"])
    print("-" * 100)


Rank: 1
Chunk ID: chunk_503
Metadata: {'document_type': 'Prospectus / Sales Literature', 'page': 2, 'source': 'care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf'}
Full text:
 which forms a part of Hospitalization. (ii) Day Care Treatment: Hospitalization involving less than 24 hours – Some surgeries doesn’t require or need not necessarily require Hospitalization Stay for minimum 24 Hours. It may be for your convenience or it may happen that the surgery underwent is minor or of intermediate complexity. We will pay through Cashless or Reimbursement Facility for all such day care treatments, maximum up to Sum Insured. (iii) Advance Technology Methods: We will indemnify you for
----------------------------------------------------------------------------------------------------

Rank: 2
Chunk ID: chunk_577
Metadata: {'page': 10, 'document_type': 'Prospectus / Sales Literature', 'source': 'care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf'

In [11]:
for qid in ["Q2", "Q3", "Q4", "Q5"]:
    print(f"\n{'='*50}")
    print(qid)

    df = all_results[qid]

    for _, row in df.iterrows():
        print("\nRank:", row["rank"])
        print("Chunk ID:", row["chunk_id"])
        print("Metadata:", row["metadata"])
        print("Full text:\n", row["text"])
        print("-" * 80)


Q2

Rank: 1
Chunk ID: chunk_503
Metadata: {'page': 2, 'source': 'care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'document_type': 'Prospectus / Sales Literature'}
Full text:
 which forms a part of Hospitalization. (ii) Day Care Treatment: Hospitalization involving less than 24 hours – Some surgeries doesn’t require or need not necessarily require Hospitalization Stay for minimum 24 Hours. It may be for your convenience or it may happen that the surgery underwent is minor or of intermediate complexity. We will pay through Cashless or Reimbursement Facility for all such day care treatments, maximum up to Sum Insured. (iii) Advance Technology Methods: We will indemnify you for
--------------------------------------------------------------------------------

Rank: 2
Chunk ID: chunk_163
Metadata: {'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'document_type': 'Prospectus / Sales Literature', 'page': 13}
Full text:
 

In [12]:
# Cell 9: Relevant chunk IDs for Care Freedom evaluation

relevant_chunk_ids = {
    "Q1": [],
    "Q2": [],
    "Q3": [],
    "Q4": ["chunk_200", "chunk_212"],
    "Q5": ["chunk_28", "chunk_74"]
}

print("Relevant chunk IDs entered successfully!")

Relevant chunk IDs entered successfully!


In [13]:
# Cell 10: Calculate Precision@5 and Recall@5

def calculate_metrics(retrieved_ids, relevant_ids, k=5):

    # If relevant chunks have not been labelled,
    # metrics cannot be calculated reliably.
    if not relevant_ids:
        return None, None

    # Take only the top-k retrieved chunks
    retrieved_top_k = retrieved_ids[:k]

    # Find relevant chunks present in the top-k results
    relevant_retrieved = set(retrieved_top_k) & set(relevant_ids)

    # Precision@k
    precision = len(relevant_retrieved) / k

    # Recall@k
    recall = len(relevant_retrieved) / len(set(relevant_ids))

    return precision, recall


# Calculate metrics for each question
metric_rows = []

for question_id, df in all_results.items():

    retrieved_ids = df["chunk_id"].tolist()
    relevant_ids = relevant_chunk_ids.get(question_id, [])

    precision, recall = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    metric_rows.append({
        "Question": question_id,
        "Precision@5": precision,
        "Recall@5": recall
    })


# Create and display results table
metrics_df = pd.DataFrame(metric_rows)

display(metrics_df)


# Calculate averages using available labelled questions
mean_precision = metrics_df["Precision@5"].mean()
mean_recall = metrics_df["Recall@5"].mean()

print("Mean Precision@5:", mean_precision)
print("Mean Recall@5:", mean_recall)

,Question,Precision@5,Recall@5
0,Q1,NaN,NaN
1,Q2,NaN,NaN
2,Q3,NaN,NaN
3,Q4,0.4,1.0
4,Q5,0.4,1.0


Mean Precision@5: 0.4
Mean Recall@5: 1.0


In [14]:
question = "What is inpatient hospitalization?"

q1_results = retrieve_chunks(question, n_results=10)

display(q1_results[["rank", "chunk_id", "metadata", "text", "distance"]])

,rank,chunk_id,metadata,text,distance
0,1,chunk_503,{'document_type': 'Prospectus / Sales Literatu...,which forms a part of Hospitalization. (ii) Da...,0.934503
1,2,chunk_577,{'source': 'care-plus--(health-insurance-produ...,It all adds up - A trip to a hospital involves...,0.939640
2,3,chunk_29,"{'document_type': 'Brochure', 'page': 3, 'sour...","and treatment at a hospital), hospitalisation ...",0.959239
3,4,chunk_72,"{'page': 2, 'source': 'care-freedom-(health-in...",to a Hospital for treatment of any One Illness...,0.978328
4,5,chunk_0,"{'page': 1, 'source': 'care--policy-usage-guid...",Policy Usage Guide In-patient Hospitalization ...,0.985742
5,6,chunk_507,{'document_type': 'Prospectus / Sales Literatu...,"Hospitalization, provided that We shall not be...",0.988583
6,7,chunk_30,{'source': 'care-freedom-(health-insurance-pro...,treatment for a period exceeding 3 consecutive...,0.994522
7,8,chunk_157,"{'page': 12, 'source': 'care-freedom-(health-i...",hospitals (iii) Any condition caused by or ass...,0.995337
8,9,chunk_502,{'source': 'care-plus--(health-insurance-produ...,"care due to Illness or Injury , which should b...",0.998545
9,10,chunk_163,"{'page': 13, 'source': 'care-freedom-(health-i...",for personal care such as help with activities...,1.004733


In [15]:
for chunk_id in ["chunk_30", "chunk_72", "chunk_0"]:
    row = q1_results[q1_results["chunk_id"] == chunk_id]

    if not row.empty:
        print("\nCHUNK ID:", chunk_id)
        print("METADATA:", row.iloc[0]["metadata"])
        print("FULL TEXT:\n", row.iloc[0]["text"])
        print("-" * 100)


CHUNK ID: chunk_30
METADATA: {'source': 'care-freedom-(health-insurance-product)---brochure.pdf', 'page': 3, 'document_type': 'Brochure'}
FULL TEXT:
 treatment for a period exceeding 3 consecutive days and had actually merited hospitalisation. CONSUMABLE ALLOWANCE It all adds up! A trip to a hospital involves more than just using the doctor’s services and hospital facilities. You are bound to incur numerous ‘non-medical’ expenses such as consumables, transportation, attendant’s cost and other daily expenses that you may not be able to even foresee. To meet these expenses, we pay ‘Consumable Allowance’ - Max. 7 days per Hospitalization covered after 3
----------------------------------------------------------------------------------------------------

CHUNK ID: chunk_72
METADATA: {'page': 2, 'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'document_type': 'Prospectus / Sales Literature'}
FULL TEXT:
 to a Hospital for treatment of any One Illne

In [16]:
care_freedom_questions = [
    "What is inpatient hospitalization?",
    "What is a day care treatment?",
    "For how many days are pre-hospitalization expenses covered?",
    "What is the coverage limit for the benefit?",
    "Is emergency ambulance expense covered?"
]

care_freedom_sources = [
    "care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf",
    "care-freedom-(health-insurance-product)---brochure.pdf"
]

care_freedom_results = {}

for i, question in enumerate(care_freedom_questions, start=1):
    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        where={"source": {"$in": care_freedom_sources}},
        include=["documents", "metadatas", "distances"]
    )

    rows = []

    for j in range(len(results["ids"][0])):
        rows.append({
            "rank": j + 1,
            "chunk_id": results["ids"][0][j],
            "metadata": results["metadatas"][0][j],
            "text": results["documents"][0][j],
            "distance": results["distances"][0][j]
        })

    care_freedom_results[f"Q{i}"] = pd.DataFrame(rows)

    print(f"\nQUESTION {i}: {question}")
    display(care_freedom_results[f"Q{i}"])


QUESTION 1: What is inpatient hospitalization?


,rank,chunk_id,metadata,text,distance
0,1,chunk_29,"{'document_type': 'Brochure', 'page': 3, 'sour...","and treatment at a hospital), hospitalisation ...",0.959239
1,2,chunk_72,"{'page': 2, 'document_type': 'Prospectus / Sal...",to a Hospital for treatment of any One Illness...,0.978328
2,3,chunk_30,"{'page': 3, 'document_type': 'Brochure', 'sour...",treatment for a period exceeding 3 consecutive...,0.994522
3,4,chunk_157,"{'page': 12, 'source': 'care-freedom-(health-i...",hospitals (iii) Any condition caused by or ass...,0.995337
4,5,chunk_163,{'source': 'care-freedom-(health-insurance-pro...,for personal care such as help with activities...,1.004733



QUESTION 2: What is a day care treatment?


,rank,chunk_id,metadata,text,distance
0,1,chunk_163,{'document_type': 'Prospectus / Sales Literatu...,for personal care such as help with activities...,0.938073
1,2,chunk_25,"{'document_type': 'Brochure', 'source': 'care-...",We also pay for your medical expenses if you u...,1.027448
2,3,chunk_68,"{'page': 2, 'source': 'care-freedom-(health-in...","charges, etc. Please refer to the Schedule of ...",1.068549
3,4,chunk_24,"{'page': 2, 'source': 'care-freedom-(health-in...","for in-patient care, for a minimum period of 2...",1.124021
4,5,chunk_47,"{'page': 6, 'source': 'care-freedom-(health-in...",PLAN OPTION In-Patient Care Day Care Treatment...,1.138120



QUESTION 3: For how many days are pre-hospitalization expenses covered?


,rank,chunk_id,metadata,text,distance
0,1,chunk_72,{'document_type': 'Prospectus / Sales Literatu...,to a Hospital for treatment of any One Illness...,0.557495
1,2,chunk_200,{'document_type': 'Prospectus / Sales Literatu...,Hospitalization expenses valid till 30 days be...,0.567939
2,3,chunk_25,"{'page': 2, 'document_type': 'Brochure', 'sour...",We also pay for your medical expenses if you u...,0.614669
3,4,chunk_75,"{'page': 3, 'document_type': 'Prospectus / Sal...",- perhaps Your state of health is such that Yo...,0.615526
4,5,chunk_212,{'document_type': 'Prospectus / Sales Literatu...,"10 days Rs.15,000 if Hospitalization exceeds 1...",0.616666



QUESTION 4: What is the coverage limit for the benefit?


,rank,chunk_id,metadata,text,distance
0,1,chunk_23,{'source': 'care-freedom-(health-insurance-pro...,if claim amount exhausts your coverage limit •...,0.900555
1,2,chunk_223,{'source': 'care-freedom-(health-insurance-pro...,in wellness centres i. Up to 8 consultations a...,0.932235
2,3,chunk_142,{'source': 'care-freedom-(health-insurance-pro...,11 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 b...,0.944595
3,4,chunk_144,{'document_type': 'Prospectus / Sales Literatu...,within 30 days from the first policy commencem...,0.959635
4,5,chunk_154,{'source': 'care-freedom-(health-insurance-pro...,for each Insured Person and Claims shall be as...,0.970045



QUESTION 5: Is emergency ambulance expense covered?


,rank,chunk_id,metadata,text,distance
0,1,chunk_28,"{'page': 3, 'source': 'care-freedom-(health-in...",COVER Getting to the hospital - It is our utmo...,0.680229
1,2,chunk_74,{'source': 'care-freedom-(health-insurance-pro...,3 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 5....,0.778727
2,3,chunk_27,"{'document_type': 'Brochure', 'source': 'care-...",POST-HOSPITALISATION Back home and till you ar...,0.794325
3,4,chunk_201,"{'page': 19, 'document_type': 'Prospectus / Sa...",19 CARE FREEDOM - UIN:CHIHLIP 26052 V 032526 A...,0.846921
4,5,chunk_213,"{'page': 20, 'source': 'care-freedom-(health-i...",End Date. Up to 10% of payable Hos- pitalizati...,0.904385


In [17]:
review_rows = []

for question_id, df in care_freedom_results.items():
    for _, row in df.iterrows():
        review_rows.append({
            "Question": question_id,
            "Rank": row["rank"],
            "Chunk_ID": row["chunk_id"],
            "Source": row["metadata"].get("source", ""),
            "Page": row["metadata"].get("page", ""),
            "Text": row["text"],
            "Relevant": ""  # Fill with Yes or No after checking
        })

review_df = pd.DataFrame(review_rows)

display(review_df)
review_df.to_csv("care_freedom_relevance_review.csv", index=False)

print("Saved as care_freedom_relevance_review.csv")

,Question,Rank,Chunk_ID,Source,Page,Text,Relevant
0,Q1,1,chunk_29,care-freedom-(health-insurance-product)---broc...,3,"and treatment at a hospital), hospitalisation ...",
1,Q1,2,chunk_72,care-freedom-(health-insurance-product)---pros...,2,to a Hospital for treatment of any One Illness...,
2,Q1,3,chunk_30,care-freedom-(health-insurance-product)---broc...,3,treatment for a period exceeding 3 consecutive...,
3,Q1,4,chunk_157,care-freedom-(health-insurance-product)---pros...,12,hospitals (iii) Any condition caused by or ass...,
4,Q1,5,chunk_163,care-freedom-(health-insurance-product)---pros...,13,for personal care such as help with activities...,
5,Q2,1,chunk_163,care-freedom-(health-insurance-product)---pros...,13,for personal care such as help with activities...,
6,Q2,2,chunk_25,care-freedom-(health-insurance-product)---broc...,2,We also pay for your medical expenses if you u...,
7,Q2,3,chunk_68,care-freedom-(health-insurance-product)---pros...,2,"charges, etc. Please refer to the Schedule of ...",
8,Q2,4,chunk_24,care-freedom-(health-insurance-product)---broc...,2,"for in-patient care, for a minimum period of 2...",
9,Q2,5,chunk_47,care-freedom-(health-insurance-product)---broc...,6,PLAN OPTION In-Patient Care Day Care Treatment...,


Saved as care_freedom_relevance_review.csv


In [21]:
# Read your relevance labels from the saved CSV
review_df = pd.read_csv("care_freedom_relevance_review.csv")

# Collect relevant chunk IDs for each question
relevant_chunk_ids = {}

for question_id in care_freedom_results.keys():
    question_rows = review_df[review_df["Question"] == question_id]

    relevant_ids = question_rows[
        question_rows["Relevant"].astype(str).str.strip().str.lower() == "yes"
    ]["Chunk_ID"].tolist()

    relevant_chunk_ids[question_id] = relevant_ids


# Calculate metrics
metric_rows = []

for question_id, df in care_freedom_results.items():
    retrieved_ids = df["chunk_id"].tolist()
    relevant_ids = relevant_chunk_ids.get(question_id, [])

    if len(relevant_ids) == 0:
        precision = float("nan")
        recall = float("nan")
    else:
        retrieved_top_k = retrieved_ids[:5]
        relevant_retrieved = set(retrieved_top_k) & set(relevant_ids)

        precision = len(relevant_retrieved) / 5
        recall = len(relevant_retrieved) / len(set(relevant_ids))

    metric_rows.append({
        "Question": question_id,
        "Precision@5": precision,
        "Recall@5": recall
    })

metrics_df = pd.DataFrame(metric_rows)

display(metrics_df)

print("Mean Precision@5:", metrics_df["Precision@5"].mean())
print("Mean Recall@5:", metrics_df["Recall@5"].mean())

,Question,Precision@5,Recall@5
0,Q1,NaN,NaN
1,Q2,NaN,NaN
2,Q3,NaN,NaN
3,Q4,NaN,NaN
4,Q5,0.4,1.0


Mean Precision@5: 0.4
Mean Recall@5: 1.0


In [19]:
print(review_df["Relevant"].value_counts(dropna=False))
display(review_df[["Question", "Chunk_ID", "Relevant"]])

Relevant
NaN    25
Name: count, dtype: int64


,Question,Chunk_ID,Relevant
0,Q1,chunk_29,NaN
1,Q1,chunk_72,NaN
2,Q1,chunk_30,NaN
3,Q1,chunk_157,NaN
4,Q1,chunk_163,NaN
5,Q2,chunk_163,NaN
6,Q2,chunk_25,NaN
7,Q2,chunk_68,NaN
8,Q2,chunk_24,NaN
9,Q2,chunk_47,NaN


In [20]:
# Mark only the chunks confirmed as relevant so far
review_df["Relevant"] = ""

confirmed_relevant = {
    "Q5": ["chunk_28", "chunk_74"]
}

for question_id, chunk_ids in confirmed_relevant.items():
    mask = (
        (review_df["Question"] == question_id) &
        (review_df["Chunk_ID"].isin(chunk_ids))
    )
    review_df.loc[mask, "Relevant"] = "Yes"

review_df.to_csv("care_freedom_relevance_review.csv", index=False)

display(review_df[["Question", "Chunk_ID", "Relevant"]])
print("Relevance labels saved.")

,Question,Chunk_ID,Relevant
0,Q1,chunk_29,
1,Q1,chunk_72,
2,Q1,chunk_30,
3,Q1,chunk_157,
4,Q1,chunk_163,
5,Q2,chunk_163,
6,Q2,chunk_25,
7,Q2,chunk_68,
8,Q2,chunk_24,
9,Q2,chunk_47,


Relevance labels saved.


In [22]:
answer_eval_df = pd.DataFrame({
    "Question": care_freedom_questions,
    "Chatbot_Answer": [""] * len(care_freedom_questions),
    "Reference_Answer": [""] * len(care_freedom_questions),
    "Faithfulness_Score_1_to_5": [""] * len(care_freedom_questions),
    "Answer_Relevance_Score_1_to_5": [""] * len(care_freedom_questions),
    "Correctness_Score_1_to_5": [""] * len(care_freedom_questions),
    "Comments": [""] * len(care_freedom_questions)
})

answer_eval_df.to_csv("answer_evaluation.csv", index=False)

display(answer_eval_df)

print("Answer evaluation sheet saved as answer_evaluation.csv")

,Question,Chatbot_Answer,Reference_Answer,Faithfulness_Score_1_to_5,Answer_Relevance_Score_1_to_5,Correctness_Score_1_to_5,Comments
0,What is inpatient hospitalization?,,,,,,
1,What is a day care treatment?,,,,,,
2,For how many days are pre-hospitalization expe...,,,,,,
3,What is the coverage limit for the benefit?,,,,,,
4,Is emergency ambulance expense covered?,,,,,,


Answer evaluation sheet saved as answer_evaluation.csv


In [23]:
answer_eval_df = pd.read_csv("answer_evaluation.csv")

score_columns = [
    "Faithfulness_Score_1_to_5",
    "Answer_Relevance_Score_1_to_5",
    "Correctness_Score_1_to_5"
]

for col in score_columns:
    answer_eval_df[col] = pd.to_numeric(
        answer_eval_df[col], errors="coerce"
    )

summary = pd.DataFrame({
    "Metric": score_columns,
    "Average Score (out of 5)": [
        answer_eval_df[col].mean() for col in score_columns
    ]
})

display(summary)

print("Overall Average Score:",
      answer_eval_df[score_columns].stack().mean())

,Metric,Average Score (out of 5)
0,Faithfulness_Score_1_to_5,NaN
1,Answer_Relevance_Score_1_to_5,NaN
2,Correctness_Score_1_to_5,NaN


Overall Average Score: nan


In [24]:
answer_eval_df = pd.read_csv("answer_evaluation.csv")

display(answer_eval_df[[
    "Question",
    "Chatbot_Answer",
    "Reference_Answer"
]])

print("\nMissing chatbot answers:",
      answer_eval_df["Chatbot_Answer"].isna().sum())

print("Missing reference answers:",
      answer_eval_df["Reference_Answer"].isna().sum())

,Question,Chatbot_Answer,Reference_Answer
0,What is inpatient hospitalization?,NaN,NaN
1,What is a day care treatment?,NaN,NaN
2,For how many days are pre-hospitalization expe...,NaN,NaN
3,What is the coverage limit for the benefit?,NaN,NaN
4,Is emergency ambulance expense covered?,NaN,NaN



Missing chatbot answers: 5
Missing reference answers: 5


In [25]:
answer_eval_df = pd.read_csv("answer_evaluation.csv")

rating_cols = [
    "Faithfulness_Score_1_to_5",
    "Answer_Relevance_Score_1_to_5",
    "Correctness_Score_1_to_5"
]

display(answer_eval_df[["Question"] + rating_cols])

print("\nMissing ratings:")
print(answer_eval_df[rating_cols].isna().sum())

,Question,Faithfulness_Score_1_to_5,Answer_Relevance_Score_1_to_5,Correctness_Score_1_to_5
0,What is inpatient hospitalization?,NaN,NaN,NaN
1,What is a day care treatment?,NaN,NaN,NaN
2,For how many days are pre-hospitalization expe...,NaN,NaN,NaN
3,What is the coverage limit for the benefit?,NaN,NaN,NaN
4,Is emergency ambulance expense covered?,NaN,NaN,NaN



Missing ratings:
Faithfulness_Score_1_to_5        5
Answer_Relevance_Score_1_to_5    5
Correctness_Score_1_to_5         5
dtype: int64


In [27]:
import os

print("Notebook working folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Notebook working folder:
C:\Data Science Course\RAG_chatbot\data

Files in this folder:
['.ipynb_checkpoints', '.virtual_documents', 'answer_evaluation.csv', 'app.py', 'care--policy-usage-guide.pdf', 'care-freedom-(health-insurance-product)---brochure.pdf', 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'care-opd---prospectus-cum-sales-literature.pdf', 'care-plus-(health-insurance-product)---brochure.pdf', 'care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'care-supreme----policy-usage-guide.pdf', 'care_freedom_relevance_review.csv', 'chroma_db', 'insurance_evalaution.ipynb', 'retrieval_results.csv', 'Untitled.ipynb']


In [29]:
import os

file_path = r"C:\Data Science Course\RAG_chatbot\data\answer_evaluation.csv"

print("File exists:", os.path.exists(file_path))
print("Current folder:", os.getcwd())
print("CSV files:", [f for f in os.listdir() if f.endswith(".csv")])

File exists: True
Current folder: C:\Data Science Course\RAG_chatbot\data
CSV files: ['answer_evaluation.csv', 'care_freedom_relevance_review.csv', 'retrieval_results.csv']


In [31]:
import pandas as pd

answer_eval_df = pd.read_csv(
    "answer_evaluation.csv",
    encoding="cp1252"
)

display(answer_eval_df.head())

,Question,Chatbot_Answer,Reference_Answer,Faithfulness_Score_1_to_5,Answer_Relevance_Score_1_to_5,Correctness_Score_1_to_5,Comments
0,What is inpatient hospitalization?,Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,4.0,5.0,4.0,NaN
1,What is a day care treatment?,A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,5.0,5.0,3.0,NaN
2,For how many days are pre-hospitalization expe...,Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,1.0,1.0,1.0,NaN
3,What is the coverage limit for the benefit?,Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,3.0,2.0,2.0,NaN
4,NaN,NaN,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,NaN,NaN


In [33]:
rating_cols = [
    "Faithfulness_Score_1_to_5",
    "Answer_Relevance_Score_1_to_5",
    "Correctness_Score_1_to_5"
]

for col in rating_cols:
    answer_eval_df[col] = pd.to_numeric(
        answer_eval_df[col], errors="coerce"
    )

print("Evaluation Results")

for col in rating_cols:
    score = answer_eval_df[col].mean()
    print(f"{col}: {score:.2f}/5" if pd.notna(score)
          else f"{col}: Missing ratings")

overall_score = answer_eval_df[rating_cols].mean().mean()

if pd.notna(overall_score):
    print(f"\nOverall Average Score: {overall_score:.2f}/5")
    print(f"Overall Percentage: {overall_score / 5 * 100:.2f}%")
else:
    print("\nOverall score cannot be calculated. Check your ratings.")

display(answer_eval_df)

Evaluation Results
Faithfulness_Score_1_to_5: 3.60/5
Answer_Relevance_Score_1_to_5: 3.60/5
Correctness_Score_1_to_5: 2.80/5

Overall Average Score: 3.33/5
Overall Percentage: 66.67%


,Question,Chatbot_Answer,Reference_Answer,Faithfulness_Score_1_to_5,Answer_Relevance_Score_1_to_5,Correctness_Score_1_to_5,Comments
0,What is inpatient hospitalization?,Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,4.0,5.0,4.0,NaN
1,What is a day care treatment?,A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,5.0,5.0,3.0,NaN
2,For how many days are pre-hospitalization expe...,Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,1.0,1.0,1.0,NaN
3,What is the coverage limit for the benefit?,Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,3.0,2.0,2.0,NaN
4,NaN,NaN,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,NaN,NaN
5,Is emergency ambulance expense covered?,"Yes, emergency ambulance expense is covered.",Yes. Care Freedom covers expenses incurred for...,5.0,5.0,4.0,NaN


In [2]:
from langchain_ollama import ChatOllama

evaluator_llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

test_response = evaluator_llm.invoke(
    "Reply with only the word: READY"
)

print(test_response.content)

READY


In [4]:
import pandas as pd

# Read CSV files with fallback encodings
def read_csv_safe(filename):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

    for encoding in encodings:
        try:
            df = pd.read_csv(filename, encoding=encoding)
            print(f"Loaded {filename} using {encoding}")
            return df
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not decode {filename}")

answer_eval_df = read_csv_safe("answer_evaluation.csv")
review_df = read_csv_safe("care_freedom_relevance_review.csv")

# Prepare data for RAGAS
ragas_rows = []

for _, row in answer_eval_df.iterrows():
    question = row["Question"]

    answer = str(row["Chatbot_Answer"]).strip()
    reference = str(row["Reference_Answer"]).strip()

    matching_chunks = review_df[
        review_df["Question"] == question
    ]["Text"].dropna().astype(str).tolist()

    ragas_rows.append({
        "question": question,
        "answer": answer,
        "contexts": matching_chunks,
        "ground_truth": reference
    })

ragas_df = pd.DataFrame(ragas_rows)

display(ragas_df)

print("\nNumber of questions:", len(ragas_df))
print("Missing answers:", (ragas_df["answer"].str.strip() == "").sum())
print("Missing reference answers:", (ragas_df["ground_truth"].str.strip() == "").sum())
print(
    "Questions with no retrieved contexts:",
    ragas_df["contexts"].apply(len).eq(0).sum()
)
    

Loaded answer_evaluation.csv using cp1252
Loaded care_freedom_relevance_review.csv using utf-8


,question,answer,contexts,ground_truth
0,What is inpatient hospitalization?,Inpatient hospitalization refers to Hospitaliz...,[],In-patient care means treatment requiring hosp...
1,What is a day care treatment?,A day-care treatment refers to hospitalization...,[],Day Care Treatment means a specified treatment...
2,For how many days are pre-hospitalization expe...,Pre-hospitalization expenses are covered for a...,[],The Prospectus states that pre-hospitalization...
3,What is the coverage limit for the benefit?,Up to 7.5% of payable Hospitalization expenses...,[],Plan 1: Up to 7.5% of payable hospitalization ...
4,NaN,nan,[],Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...
5,Is emergency ambulance expense covered?,"Yes, emergency ambulance expense is covered.",[],Yes. Care Freedom covers expenses incurred for...



Number of questions: 6
Missing answers: 0
Missing reference answers: 0
Questions with no retrieved contexts: 6


In [5]:
import ragas

print("RAGAS version:", ragas.__version__)

RAGAS version: 0.4.3


In [7]:
%pip install -U langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(
    ChatOllama(
        model="llama3.2",
        temperature=0
    )
)

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

print("RAGAS evaluator setup complete.")

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\3668946861.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAGAS evaluator setup complete.


C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\3668946861.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


In [3]:
import pandas as pd

# Load the CSV files
answer_eval_df = pd.read_csv(
    "answer_evaluation.csv",
    encoding="cp1252"
)

review_df = pd.read_csv(
    "care_freedom_relevance_review.csv",
    encoding="latin1"
)

# Recreate the RAGAS dataset
ragas_rows = []

for _, row in answer_eval_df.iterrows():
    question = str(row["Question"]).strip()
    answer = str(row["Chatbot_Answer"]).strip()
    reference = str(row["Reference_Answer"]).strip()

    contexts = review_df.loc[
        review_df["Question"].astype(str).str.strip() == question,
        "Text"
    ].dropna().astype(str).tolist()

    ragas_rows.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": reference
    })

ragas_df = pd.DataFrame(ragas_rows)

print("RAGAS dataframe recreated.")
print("Total questions:", len(ragas_df))
display(ragas_df)

RAGAS dataframe recreated.
Total questions: 6


,question,answer,contexts,ground_truth
0,What is inpatient hospitalization?,Inpatient hospitalization refers to Hospitaliz...,[],In-patient care means treatment requiring hosp...
1,What is a day care treatment?,A day-care treatment refers to hospitalization...,[],Day Care Treatment means a specified treatment...
2,For how many days are pre-hospitalization expe...,Pre-hospitalization expenses are covered for a...,[],The Prospectus states that pre-hospitalization...
3,What is the coverage limit for the benefit?,Up to 7.5% of payable Hospitalization expenses...,[],Plan 1: Up to 7.5% of payable hospitalization ...
4,nan,nan,[],Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...
5,Is emergency ambulance expense covered?,"Yes, emergency ambulance expense is covered.",[],Yes. Care Freedom covers expenses incurred for...


In [4]:
from ragas import SingleTurnSample, EvaluationDataset

samples = []

for _, row in ragas_df.iterrows():
    samples.append(
        SingleTurnSample(
            user_input=str(row["question"]),
            response=str(row["answer"]),
            retrieved_contexts=list(row["contexts"]),
            reference=str(row["ground_truth"])
        )
    )

eval_dataset = EvaluationDataset(samples=samples)

print("RAGAS dataset ready!")
print("Total samples:", len(eval_dataset.samples))

RAGAS dataset ready!
Total samples: 6


In [5]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall
)

# Initialize the evaluation metrics
metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall()
]

# Run RAGAS evaluation
ragas_result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

# Display the results
ragas_scores_df = ragas_result.to_pandas()
display(ragas_scores_df)

print("\nAverage RAGAS Scores:")
display(ragas_scores_df[
    ["faithfulness", "answer_relevancy",
     "llm_context_precision_with_reference", "context_recall"]
].mean(numeric_only=True))

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\3689922717.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\3689922717.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\3689922717.py:2: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

Exception raised in Job[17]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[21]: OutputParserException(Invalid json output: Does emergency ambulance expense get covered?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What is inpatient hospitalization?,[],Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,NaN,NaN,0.0,0.666667
1,What is a day care treatment?,[],A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,NaN,NaN,0.0,NaN
2,For how many days are pre-hospitalization expe...,[],Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,NaN,NaN,0.0,1.000000
3,What is the coverage limit for the benefit?,[],Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,NaN,NaN,0.0,1.000000
4,nan,[],nan,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,0.0,NaN
5,Is emergency ambulance expense covered?,[],"Yes, emergency ambulance expense is covered.",Yes. Care Freedom covers expenses incurred for...,NaN,NaN,0.0,1.000000



Average RAGAS Scores:


faithfulness                                 NaN
answer_relevancy                             NaN
llm_context_precision_with_reference    0.000000
context_recall                          0.916667
dtype: float64

In [6]:
# Save individual question-wise RAGAS scores
ragas_scores_df.to_csv("ragas_evaluation_results.csv", index=False)

# Calculate average score for each available numeric metric
ragas_summary = ragas_scores_df.select_dtypes(include="number").mean()

print("RAGAS results saved to ragas_evaluation_results.csv")
print("\nAverage RAGAS Scores:")
display(ragas_summary.to_frame(name="Average Score"))

RAGAS results saved to ragas_evaluation_results.csv

Average RAGAS Scores:


,Average Score
faithfulness,NaN
answer_relevancy,NaN
llm_context_precision_with_reference,0.000000
context_recall,0.916667


In [7]:
print("Question-wise RAGAS scores:")

score_columns = [
    col for col in ragas_scores_df.columns
    if col in [
        "question",
        "faithfulness",
        "answer_relevancy",
        "llm_context_precision_with_reference",
        "context_recall"
    ]
]

display(ragas_scores_df[score_columns])

print("\nMissing values per metric:")
display(ragas_scores_df[[
    "faithfulness",
    "answer_relevancy",
    "llm_context_precision_with_reference",
    "context_recall"
]].isna().sum())

Question-wise RAGAS scores:


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,NaN,NaN,0.0,0.666667
1,NaN,NaN,0.0,NaN
2,NaN,NaN,0.0,1.000000
3,NaN,NaN,0.0,1.000000
4,NaN,NaN,0.0,NaN
5,NaN,NaN,0.0,1.000000



Missing values per metric:


faithfulness                            6
answer_relevancy                        6
llm_context_precision_with_reference    0
context_recall                          2
dtype: int64

In [10]:
print("Total rows:", len(ragas_scores_df))

print("\nAvailable columns:")
print(ragas_scores_df.columns.tolist())

print("\nFirst 5 rows:")
display(ragas_scores_df.head())

print("\nContext count for each question:")
display(
    ragas_df[["question", "contexts"]].assign(
        context_count=ragas_df["contexts"].apply(len)
    )[["question", "context_count"]]
)

Total rows: 6

Available columns:
['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'llm_context_precision_with_reference', 'context_recall']

First 5 rows:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What is inpatient hospitalization?,[],Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,NaN,NaN,0.0,0.666667
1,What is a day care treatment?,[],A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,NaN,NaN,0.0,NaN
2,For how many days are pre-hospitalization expe...,[],Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,NaN,NaN,0.0,1.000000
3,What is the coverage limit for the benefit?,[],Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,NaN,NaN,0.0,1.000000
4,nan,[],nan,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,0.0,NaN



Context count for each question:


,question,context_count
0,What is inpatient hospitalization?,0
1,What is a day care treatment?,0
2,For how many days are pre-hospitalization expe...,0
3,What is the coverage limit for the benefit?,0
4,nan,0
5,Is emergency ambulance expense covered?,0


In [11]:
metrics = [
    "faithfulness",
    "answer_relevancy",
    "llm_context_precision_with_reference",
    "context_recall"
]

# Keep only metrics that exist in the results
available_metrics = [
    metric for metric in metrics
    if metric in ragas_scores_df.columns
]

# Convert metric columns to numeric values
for metric in available_metrics:
    ragas_scores_df[metric] = pd.to_numeric(
        ragas_scores_df[metric], errors="coerce"
    )

summary = pd.DataFrame({
    "Average Score": ragas_scores_df[available_metrics].mean(),
    "Valid Scores": ragas_scores_df[available_metrics].count(),
    "Missing Scores": ragas_scores_df[available_metrics].isna().sum()
})

display(summary)

summary.to_csv("ragas_summary.csv")
print("Summary saved as ragas_summary.csv")

,Average Score,Valid Scores,Missing Scores
faithfulness,NaN,0,6
answer_relevancy,NaN,0,6
llm_context_precision_with_reference,0.000000,6,0
context_recall,0.916667,4,2


Summary saved as ragas_summary.csv


In [12]:
print("Number of evaluation samples:", len(ragas_df))

print("\nContexts per question:")
for _, row in ragas_df.iterrows():
    print(f"\nQuestion: {row['question']}")
    print("Context count:", len(row["contexts"]))

    if len(row["contexts"]) == 0:
        print("WARNING: No retrieved context found for this question.")
    else:
        print("First context preview:")
        print(str(row["contexts"][0])[:300])

Number of evaluation samples: 6

Contexts per question:

Question: What is inpatient hospitalization?
Context count: 0

Question: What is a day care treatment?
Context count: 0

Question: For how many days are pre-hospitalization expenses covered?
Context count: 0

Question: What is the coverage limit for the benefit?
Context count: 0

Question: nan
Context count: 0

Question: Is emergency ambulance expense covered?
Context count: 0


In [13]:
print("RAGAS result object:")
print(ragas_result)

print("\nResult dataframe columns:")
print(ragas_scores_df.columns.tolist())

print("\nResult dataframe shape:")
print(ragas_scores_df.shape)

print("\nFirst rows:")
display(ragas_scores_df.head())

RAGAS result object:
{'faithfulness': nan, 'answer_relevancy': nan, 'llm_context_precision_with_reference': 0.0000, 'context_recall': 0.9167}

Result dataframe columns:
['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'llm_context_precision_with_reference', 'context_recall']

Result dataframe shape:
(6, 8)

First rows:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What is inpatient hospitalization?,[],Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,NaN,NaN,0.0,0.666667
1,What is a day care treatment?,[],A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,NaN,NaN,0.0,NaN
2,For how many days are pre-hospitalization expe...,[],Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,NaN,NaN,0.0,1.000000
3,What is the coverage limit for the benefit?,[],Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,NaN,NaN,0.0,1.000000
4,nan,[],nan,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,0.0,NaN


In [15]:
print("ChromaDB-related variables currently available:")

for name in ["collection", "chroma_collection", "client", "chroma_client"]:
    print(name, "->", name in globals())

ChromaDB-related variables currently available:
collection -> False
chroma_collection -> False
client -> False
chroma_client -> False


In [16]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_collection(
    name="insurance_documents"
)

print("Connected to ChromaDB!")
print("Number of stored chunks:", collection.count())

Connected to ChromaDB!
Number of stored chunks: 872


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

def retrieve_chunks(question, n_results=5):
    query_embedding = embedding_model.embed_query(question)

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    return result["documents"][0] if result["documents"] else []

# Get fresh retrieval contexts for each evaluation question
ragas_df["contexts"] = ragas_df["question"].apply(
    lambda question: retrieve_chunks(question, n_results=5)
)

print("Questions:", len(ragas_df))
print("Questions with retrieved contexts:",
      sum(bool(contexts) for contexts in ragas_df["contexts"]))

display(ragas_df[["question", "contexts"]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Questions: 6
Questions with retrieved contexts: 6


,question,contexts
0,What is inpatient hospitalization?,[which forms a part of Hospitalization. (ii) D...
1,What is a day care treatment?,[which forms a part of Hospitalization. (ii) D...
2,For how many days are pre-hospitalization expe...,[for expenses incurred under Benefit 'In-patie...
3,What is the coverage limit for the benefit?,[if claim amount exhausts your coverage limit ...
4,nan,"[3,00,000 28,757 5% 3,00,000 50,492 NA 50,492 ..."
5,Is emergency ambulance expense covered?,"[fees, prescribed diagnostics or prescribed me..."


In [18]:
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall
)

# Create evaluation samples using the freshly retrieved contexts
samples = []

for _, row in ragas_df.iterrows():
    samples.append(
        SingleTurnSample(
            user_input=str(row["question"]),
            response=str(row["answer"]),
            retrieved_contexts=list(row["contexts"]),
            reference=str(row["ground_truth"])
        )
    )

eval_dataset = EvaluationDataset(samples=samples)

# Evaluate
metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall()
]

ragas_result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

ragas_scores_df = ragas_result.to_pandas()

# Display individual results
display(ragas_scores_df)

# Display average scores
metric_columns = [
    "faithfulness",
    "answer_relevancy",
    "llm_context_precision_with_reference",
    "context_recall"
]

print("Average RAGAS Scores:")
display(ragas_scores_df[metric_columns].mean(numeric_only=True))

# Save results
ragas_scores_df.to_csv("ragas_evaluation_results.csv", index=False)
ragas_scores_df[metric_columns].mean(numeric_only=True).to_csv(
    "ragas_summary.csv",
    header=["Average Score"]
)

print("Saved: ragas_evaluation_results.csv and ragas_summary.csv")

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\1290111307.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\1290111307.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\1290111307.py:2: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[21]: TimeoutError()
Exception raised in Job[22]: TimeoutError(

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What is inpatient hospitalization?,[which forms a part of Hospitalization. (ii) D...,Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,NaN,NaN,NaN,NaN
1,What is a day care treatment?,[which forms a part of Hospitalization. (ii) D...,A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,NaN,NaN,NaN,NaN
2,For how many days are pre-hospitalization expe...,[for expenses incurred under Benefit 'In-patie...,Pre-hospitalization expenses are covered for a...,The Prospectus states that pre-hospitalization...,NaN,NaN,NaN,NaN
3,What is the coverage limit for the benefit?,[if claim amount exhausts your coverage limit ...,Up to 7.5% of payable Hospitalization expenses...,Plan 1: Up to 7.5% of payable hospitalization ...,NaN,NaN,NaN,NaN
4,nan,"[3,00,000 28,757 5% 3,00,000 50,492 NA 50,492 ...",nan,Plan 2: Up to 7.5% for ?2–4 lakh Sum Insured; ...,NaN,NaN,NaN,NaN
5,Is emergency ambulance expense covered?,"[fees, prescribed diagnostics or prescribed me...","Yes, emergency ambulance expense is covered.",Yes. Care Freedom covers expenses incurred for...,NaN,NaN,NaN,NaN


Average RAGAS Scores:


faithfulness                           NaN
answer_relevancy                       NaN
llm_context_precision_with_reference   NaN
context_recall                         NaN
dtype: float64

Saved: ragas_evaluation_results.csv and ragas_summary.csv


In [21]:
from langchain_ollama import ChatOllama

test_llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = test_llm.invoke("Reply with only: READY")
print(response.content)

READY


In [22]:
from ragas import evaluate
from ragas.metrics import Faithfulness
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=600,
    max_retries=1,
    max_workers=1
)

test_result = evaluate(
    dataset=test_dataset,
    metrics=[Faithfulness()],
    llm=evaluator_llm,
    run_config=run_config
)

test_scores_df = test_result.to_pandas()
display(test_scores_df)

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\4067087377.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness
0,What is inpatient hospitalization?,[which forms a part of Hospitalization. (ii) D...,Inpatient hospitalization refers to Hospitaliz...,In-patient care means treatment requiring hosp...,1.0
1,What is a day care treatment?,[which forms a part of Hospitalization. (ii) D...,A day-care treatment refers to hospitalization...,Day Care Treatment means a specified treatment...,1.0


In [23]:
test_scores_df = test_result.to_pandas()

print("Faithfulness score:")
display(test_scores_df[["faithfulness"]])

print("Average Faithfulness:")
display(test_scores_df["faithfulness"].mean())

Faithfulness score:


,faithfulness
0,1.0
1,1.0


Average Faithfulness:


np.float64(1.0)

In [24]:
from ragas import evaluate
from ragas.metrics import AnswerRelevancy
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=600,
    max_retries=1,
    max_workers=1
)

answer_relevancy_result = evaluate(
    dataset=test_dataset,
    metrics=[AnswerRelevancy()],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=run_config
)

answer_relevancy_df = answer_relevancy_result.to_pandas()

display(answer_relevancy_df[["answer_relevancy"]])
print("Average Answer Relevancy:",
      answer_relevancy_df["answer_relevancy"].mean())

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\617595683.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Exception raised in Job[0]: OutputParserException(Invalid json output: Inpatient hospitalization refers to Hospitalization involving more than 24 hours.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[1]: OutputParserException(Invalid json output: A day-care treatment refers to hospitalization involving less than 24 hours
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


,answer_relevancy
0,NaN
1,NaN


Average Answer Relevancy: nan


In [25]:
from ragas import evaluate
from ragas.metrics import AnswerRelevancy
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=600,
    max_retries=1,
    max_workers=1
)

try:
    debug_result = evaluate(
        dataset=test_dataset,
        metrics=[AnswerRelevancy()],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=run_config,
        raise_exceptions=True
    )

    debug_df = debug_result.to_pandas()
    display(debug_df)

except Exception as e:
    print("Answer Relevancy error:")
    print(type(e).__name__)
    print(str(e))

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_31112\2719958511.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Answer Relevancy error:
OutputParserException
Invalid json output: Given the input:
{
    "response": "Inpatient hospitalization refers to Hospitalization involving more than 24 hours."
}

The generated question is:
"What does inpatient hospitalization refer to?"

The answer is committal, so the output in JSON format is:

{
    "question": "What does inpatient hospitalization refer to?",
    "noncommittal": 0
}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
